# 03. Factual Consistency and Structured Qualitative Evaluation

This notebook builds a **structured qualitative evaluation** for summarization outputs. The goal is to evaluate the same fixed sample of articles across multiple methods using a consistent rubric.

This is primarily a **human qualitative review**, not an automatic metric. Code is used to:

1. select a fixed sample of 30 examples,
2. organize each article, reference summary, and model output into a scoring workbook,
3. define the rubric clearly,
4. summarize method-level results after scores are entered.

Evaluation dimensions:

- Fluency
- Factual consistency
- Coverage
- Conciseness


In [1]:
!pip -q install openpyxl openai

In [2]:
import pandas as pd
from pathlib import Path

USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

RUN_DIR = Path('/content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2')
RESULTS_DIR = RUN_DIR / 'results'
EXT_RESULTS_DIR = RESULTS_DIR / 'extension_long_context'

# Notebook 02 evaluates all long articles from the same 1,500-example test set.
# This file contains 440 examples with article length >1024 BART tokens.
LONG_REVIEW_PATH = EXT_RESULTS_DIR / 'long_article_comparison_for_factual_review_440.csv'
MAIN_PREDICTIONS_PATH = RESULTS_DIR / 'test_predictions_with_lengths.csv'

OUTPUT_XLSX = RESULTS_DIR / 'manual_factual_consistency_rubric_30_examples.xlsx'
OUTPUT_CSV = RESULTS_DIR / 'manual_factual_consistency_rubric_30_examples.csv'
SAMPLE_IDS_PATH = RESULTS_DIR / 'manual_factual_consistency_sample_ids_30.csv'
SUMMARY_PATH = RESULTS_DIR / 'manual_qualitative_rubric_summary.csv'
SEED = 42

# Instructor requested a fixed sample of at least 20-30 examples.
MANUAL_EVAL_ARTICLES = 30


Mounted at /content/drive


## Rubric Definition

Use 1-5 scores for each model output.

| Dimension | 5 | 3 | 1 |
|---|---|---|---|
| Fluency | Natural, grammatical, easy to read | Understandable but awkward | Hard to read or incoherent |
| Factual consistency | Fully supported by the source article | Mostly supported with minor ambiguity | Contradiction or hallucinated claim |
| Coverage | Captures the main reference information | Captures some important points | Misses the central information |
| Conciseness | Focused and summary-like | Some unnecessary detail | Too verbose, repetitive, or unfocused |

Suggested error labels: `none`, `omission`, `hallucination`, `entity_error`, `relation_error`, `over_specific`, `too_verbose`, `unclear`.

## Create Structured Qualitative Evaluation Workbook

The workbook uses the long-article comparison file from Notebook 02 when available, because it contains multiple model outputs on the same source articles.

By default, we use a **fixed sample of 30 articles**. Each article has one row per method, so with five methods the workbook has 150 rows. The same 30 articles are used for every model, which makes the method comparison fair.

The score columns are intentionally blank. They should be filled by a human reviewer using the rubric above. This matches the instructor's request for a structured qualitative rubric; it is not meant to be another automatic metric like ROUGE or BERTScore.


In [3]:
MODEL_COLUMNS = [
    ('Lead-3', 'lead3_prediction'),
    ('Fine-tuned BART-base, 1024-token truncation', 'bart_truncated_prediction'),
    ('Hierarchical BART', 'hierarchical_bart_prediction'),
    ('Fine-tuned LED-base-16384 CNN/DM', 'fine_tuned_led_base_16384_cnn_dm_prediction'),
    ('BART-large-CNN reference', 'bart_large_cnn_reference_prediction'),
]

if LONG_REVIEW_PATH.exists():
    source_df = pd.read_csv(LONG_REVIEW_PATH)
    source_label = 'long_article_extension_440'
else:
    source_df = pd.read_csv(MAIN_PREDICTIONS_PATH).rename(columns={'finetuned_bart_prediction': 'bart_truncated_prediction'})
    MODEL_COLUMNS = [('Lead-3', 'lead3_prediction'), ('Fine-tuned BART-base, 1024-token truncation', 'bart_truncated_prediction')]
    source_label = 'main_test_predictions'

available_model_columns = [(name, col) for name, col in MODEL_COLUMNS if col in source_df.columns]
sample_n = min(MANUAL_EVAL_ARTICLES, len(source_df))
sample = source_df.sample(n=sample_n, random_state=SEED).reset_index(drop=True)

def clean_text(text):
    return str(text).replace('\\r', ' ').replace('\\n', ' ').strip()
sample_ids = sample[['test_row_id', 'article_tokens']].copy() if {'test_row_id', 'article_tokens'}.issubset(sample.columns) else sample.reset_index()[['index']].rename(columns={'index': 'source_row'})
sample_ids.insert(0, 'qualitative_example_id', range(1, len(sample_ids) + 1))
sample_ids['source_file'] = source_label
sample_ids['seed'] = SEED
sample_ids.to_csv(SAMPLE_IDS_PATH, index=False)

rows = []
for example_id, row in sample.iterrows():
    for model_name, pred_col in available_model_columns:
        rows.append({
            'qualitative_example_id': example_id + 1,
            'test_row_id': row.get('test_row_id', ''),
            'article_tokens': row.get('article_tokens', ''),
            'source_article': clean_text(row['article']),
            'reference_summary': clean_text(row['reference']),
            'model': model_name,
            'model_summary': clean_text(row[pred_col]),
            'fluency_1to5': '',
            'factual_consistency_1to5': '',
            'coverage_1to5': '',
            'conciseness_1to5': '',
            'primary_error_type': '',
            'notes': '',
        })

rubric_df = pd.DataFrame(rows)
rubric_df.to_csv(OUTPUT_CSV, index=False)

rubric_guide = pd.DataFrame([
    ['Fluency', 'Natural, grammatical, easy to read', 'Mostly fluent with small awkward phrases', 'Understandable but awkward', 'Difficult to read in several places', 'Hard to read or incoherent'],
    ['Factual consistency', 'All claims are supported by the source article', 'Almost all claims supported; very minor ambiguity', 'Mostly supported but includes ambiguity or weakly grounded detail', 'Contains a clear unsupported or distorted claim', 'Contradiction, hallucinated claim, or wrong entity/relation'],
    ['Coverage', 'Captures the central reference information', 'Captures most major points with small omissions', 'Captures some important points but misses others', 'Misses most important information', 'Misses the central event or main takeaway'],
    ['Conciseness', 'Focused and summary-like', 'Mostly focused with minor extra detail', 'Some unnecessary detail', 'Too verbose, repetitive, or unfocused', 'Not summary-like; mostly irrelevant or rambling'],
], columns=['dimension', 'score_5', 'score_4', 'score_3', 'score_2', 'score_1'])

scoring_instructions = pd.DataFrame([
    ['Unit of evaluation', 'Score each row independently: one article, one reference summary, one model output.'],
    ['Fixed sample', f'The workbook contains {sample_n} fixed articles selected with random seed {SEED}. Every method is evaluated on the same articles.'],
    ['Score scale', 'Use integers from 1 to 5 for fluency, factual consistency, coverage, and conciseness.'],
    ['Factual consistency rule', 'Check whether model-summary claims are supported by the source article, not whether they merely match the reference wording.'],
    ['Coverage rule', 'Use the reference summary as guidance for what information matters, but allow valid paraphrases.'],
    ['Conciseness rule', 'Penalize summaries that are too verbose, repetitive, or include unnecessary details.'],
    ['Primary error type', 'Choose the main error when one exists: none, omission, hallucination, entity_error, relation_error, over_specific, too_verbose, unclear.'],
    ['Why human scoring', 'These dimensions require judgment. Automatic metrics can support the analysis but should not replace the qualitative rubric.'],
], columns=['item', 'instruction'])

model_key = pd.DataFrame(available_model_columns, columns=['model', 'prediction_column'])
model_key['source_file'] = source_label

method_expectation_guide = pd.DataFrame([
    {
        'method': 'Lead-3',
        'expected_strength': 'Often factually safe because it copies opening sentences.',
        'expected_weakness': 'May include irrelevant lead details and miss information outside the first few sentences.',
        'what_to_watch_in_scores': 'High factual consistency, variable coverage and conciseness.'
    },
    {
        'method': 'Fine-tuned BART-base, 1024-token truncation',
        'expected_strength': 'Usually more abstractive and summary-like than Lead-3.',
        'expected_weakness': 'Can omit information that appears after the 1024-token cutoff or add unsupported details.',
        'what_to_watch_in_scores': 'Often stronger fluency and conciseness; factuality and coverage need checking.'
    },
    {
        'method': 'Hierarchical BART',
        'expected_strength': 'Can access later article sections through chunk summaries.',
        'expected_weakness': 'Chunk summaries can accumulate noise or propagate early compression errors.',
        'what_to_watch_in_scores': 'May improve coverage in some cases but lose conciseness or factual precision.'
    },
    {
        'method': 'Fine-tuned LED-base-16384 CNN/DM',
        'expected_strength': 'Long-context architecture can read more of the source article.',
        'expected_weakness': 'Generation may be less aligned with concise CNN/DailyMail highlight style in this setup.',
        'what_to_watch_in_scores': 'Coverage may improve for tail facts, but fluency/conciseness/factuality can vary.'
    },
    {
        'method': 'BART-large-CNN reference',
        'expected_strength': 'Strong task-specific external reference model.',
        'expected_weakness': 'Not our trained model and not a long-context model.',
        'what_to_watch_in_scores': 'Useful upper reference for fluency, coverage, and conciseness.'
    },
])

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    rubric_df.to_excel(writer, sheet_name='manual_scoring', index=False)
    rubric_guide.to_excel(writer, sheet_name='rubric_guide', index=False)
    scoring_instructions.to_excel(writer, sheet_name='scoring_instructions', index=False)
    model_key.to_excel(writer, sheet_name='model_key', index=False)
    method_expectation_guide.to_excel(writer, sheet_name='method_expectations', index=False)
    sample_ids.to_excel(writer, sheet_name='fixed_sample_ids', index=False)
    ws = writer.sheets['manual_scoring']
    ws.freeze_panes = 'A2'
    widths = {'A':20,'B':12,'C':14,'D':70,'E':55,'F':34,'G':60,'H':14,'I':24,'J':14,'K':16,'L':20,'M':40}
    for col, width in widths.items():
        ws.column_dimensions[col].width = width
    for row_cells in ws.iter_rows():
        for cell in row_cells:
            cell.alignment = cell.alignment.copy(wrap_text=True, vertical='top')

print('Saved structured qualitative evaluation workbook:', OUTPUT_XLSX)
print('Saved CSV copy:', OUTPUT_CSV)
print('Saved fixed sample IDs:', SAMPLE_IDS_PATH)
print('Rows:', len(rubric_df), 'Articles:', rubric_df['qualitative_example_id'].nunique(), 'Methods:', rubric_df['model'].nunique())
display(rubric_df.head(10))


Saved structured qualitative evaluation workbook: /content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2/results/manual_factual_consistency_rubric_30_examples.xlsx
Saved CSV copy: /content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2/results/manual_factual_consistency_rubric_30_examples.csv
Saved fixed sample IDs: /content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2/results/manual_factual_consistency_sample_ids_30.csv
Rows: 150 Articles: 30 Methods: 5


/tmp/ipykernel_985/1273782491.py:119: DeprecationWarning: Call to deprecated function copy (Use copy(obj) or cell.obj = cell.obj + other).
  cell.alignment = cell.alignment.copy(wrap_text=True, vertical='top')


,qualitative_example_id,test_row_id,article_tokens,source_article,reference_summary,model,model_summary,fluency_1to5,factual_consistency_1to5,coverage_1to5,conciseness_1to5,primary_error_type,notes
0,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,Lead-3,Women have long been warned that the older the...,,,,,,
1,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,"Fine-tuned BART-base, 1024-token truncation","Two-thirds of new UK fathers are now over 30, ...",,,,,,
2,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,Hierarchical BART,"Two-thirds of new UK fathers are now over 30, ...",,,,,,
3,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,Fine-tuned LED-base-16384 CNN/DM,men over 35 have a 50% lower chance of conceiv...,,,,,,
4,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,BART-large-CNN reference,"Two-thirds of new UK fathers are now over 30, ...",,,,,,
5,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,Lead-3,The mother who crashed a car into a lake near ...,,,,,,
6,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,"Fine-tuned BART-base, 1024-token truncation","Akon Guode, 35, was behind the wheel of a grey...",,,,,,
7,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,Hierarchical BART,"Akon Guode, 35, was behind the wheel of a grey...",,,,,,
8,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,Fine-tuned LED-base-16384 CNN/DM,the four-wheel-drive plunged into the lake at ...,,,,,,
9,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,BART-large-CNN reference,The mother who crashed a car into a lake near ...,,,,,,


## DeepSeek-as-a-Judge Automatic Rubric Scoring

This section uses DeepSeek as an automatic judge to apply the same rubric to the fixed 30-example sample. It scores each model output on fluency, factual consistency, coverage, and conciseness.

Important: this is an **LLM-as-a-judge evaluation**, not a replacement for human review. In the report, describe it as an automatic qualitative scoring aid and mention that LLM judgments can be biased or imperfect.


In [4]:
import os
import json
import time
from getpass import getpass
from openai import OpenAI

DEEPSEEK_MODEL = 'deepseek-chat'
DEEPSEEK_BASE_URL = 'https://api.deepseek.com'
DEEPSEEK_SCORED_CSV = RESULTS_DIR / 'deepseek_scored_qualitative_rubric_30_examples.csv'
DEEPSEEK_SCORED_XLSX = RESULTS_DIR / 'deepseek_scored_qualitative_rubric_30_examples.xlsx'
DEEPSEEK_METHOD_SUMMARY_PATH = RESULTS_DIR / 'deepseek_method_comparison_summary.csv'
DEEPSEEK_ERROR_SUMMARY_PATH = RESULTS_DIR / 'deepseek_error_type_summary.csv'
MAX_ARTICLE_CHARS = 20000
SLEEP_BETWEEN_CALLS = 0.5

api_key = os.getenv('DEEPSEEK_API_KEY')
try:
    from google.colab import userdata
    api_key = api_key or userdata.get('DEEPSEEK_API_KEY')
except Exception:
    pass

if not api_key:
    api_key = getpass('Enter DeepSeek API key: ')

client = OpenAI(api_key=api_key, base_url=DEEPSEEK_BASE_URL)

score_cols = ['fluency_1to5', 'factual_consistency_1to5', 'coverage_1to5', 'conciseness_1to5']
valid_error_types = {'none', 'omission', 'hallucination', 'entity_error', 'relation_error', 'over_specific', 'too_verbose', 'unclear'}

def extract_json_object(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.strip('`')
        if text.lower().startswith('json'):
            text = text[4:].strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find('{')
        end = text.rfind('}')
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise

def normalize_score(value):
    try:
        value = int(round(float(value)))
    except Exception:
        value = 3
    return max(1, min(5, value))

def judge_one_article(article_rows):
    first = article_rows.iloc[0]
    payload = {
        'article': str(first['source_article'])[:MAX_ARTICLE_CHARS],
        'reference_summary': str(first['reference_summary']),
        'model_outputs': [
            {'model': str(row['model']), 'summary': str(row['model_summary'])}
            for _, row in article_rows.iterrows()
        ]
    }
    system_prompt = 'You are a strict but fair evaluator for news summarization. Apply the rubric consistently across models. Return valid JSON only.'
    user_prompt = (
        'Evaluate each model summary against the source article and the reference summary.\n\n'
        'Use integer scores from 1 to 5:\n'
        '- fluency_1to5: grammaticality and readability.\n'
        '- factual_consistency_1to5: whether claims are supported by the source article.\n'
        '- coverage_1to5: whether the summary captures the main reference-relevant information.\n'
        '- conciseness_1to5: whether the summary is focused and not verbose/noisy.\n\n'
        'Choose one primary_error_type from: none, omission, hallucination, entity_error, relation_error, over_specific, too_verbose, unclear.\n\n'
        'Return this exact JSON schema:\n'
        '{"scores":[{"model":"model name","fluency_1to5":1,"factual_consistency_1to5":1,"coverage_1to5":1,"conciseness_1to5":1,"primary_error_type":"none","rationale":"brief reason, 1-2 sentences"}]}\n\n'
        'Data:\n' + json.dumps(payload, ensure_ascii=False)
    )
    kwargs = dict(
        model=DEEPSEEK_MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )
    try:
        response = client.chat.completions.create(**kwargs, response_format={'type': 'json_object'})
    except TypeError:
        response = client.chat.completions.create(**kwargs)
    content = response.choices[0].message.content
    parsed = extract_json_object(content)
    return parsed.get('scores', [])


Enter DeepSeek API key: ··········


In [5]:
rubric_df = pd.read_csv(OUTPUT_CSV)
completed_ids = set()
all_results = []

if DEEPSEEK_SCORED_CSV.exists():
    existing = pd.read_csv(DEEPSEEK_SCORED_CSV)
    all_results = existing.to_dict('records')
    completed_ids = set(existing['qualitative_example_id'].dropna().astype(int).unique())
    print(f'Resuming from existing file. Completed examples: {len(completed_ids)}')

for example_id, article_rows in rubric_df.groupby('qualitative_example_id', sort=True):
    example_id = int(example_id)
    if example_id in completed_ids:
        continue

    print(f'Judging qualitative example {example_id}/{rubric_df["qualitative_example_id"].nunique()} ...')
    try:
        scores = judge_one_article(article_rows)
    except Exception as exc:
        print(f'Failed on example {example_id}: {exc}')
        raise

    score_by_model = {str(item.get('model')): item for item in scores}
    for _, row in article_rows.iterrows():
        model = str(row['model'])
        item = score_by_model.get(model, {})
        result = row.to_dict()
        for col in score_cols:
            result[col] = normalize_score(item.get(col, 3))
        error_type = str(item.get('primary_error_type', 'unclear')).strip()
        if error_type not in valid_error_types:
            error_type = 'unclear'
        result['primary_error_type'] = error_type
        result['deepseek_rationale'] = str(item.get('rationale', '')).strip()
        result['judge_model'] = DEEPSEEK_MODEL
        all_results.append(result)

    pd.DataFrame(all_results).to_csv(DEEPSEEK_SCORED_CSV, index=False)
    time.sleep(SLEEP_BETWEEN_CALLS)

scored_df = pd.DataFrame(all_results)
scored_df.to_csv(DEEPSEEK_SCORED_CSV, index=False)
with pd.ExcelWriter(DEEPSEEK_SCORED_XLSX, engine='openpyxl') as writer:
    scored_df.to_excel(writer, sheet_name='deepseek_scores', index=False)
    rubric_guide.to_excel(writer, sheet_name='rubric_guide', index=False)
    scoring_instructions.to_excel(writer, sheet_name='scoring_instructions', index=False)

print('Saved DeepSeek row-level scores:', DEEPSEEK_SCORED_CSV)
print('Saved DeepSeek Excel scores:', DEEPSEEK_SCORED_XLSX)
print('Rows:', len(scored_df), 'Articles:', scored_df['qualitative_example_id'].nunique(), 'Methods:', scored_df['model'].nunique())
display(scored_df.head(10))


Judging qualitative example 1/30 ...
Judging qualitative example 2/30 ...
Judging qualitative example 3/30 ...
Judging qualitative example 4/30 ...
Judging qualitative example 5/30 ...
Judging qualitative example 6/30 ...
Judging qualitative example 7/30 ...
Judging qualitative example 8/30 ...
Judging qualitative example 9/30 ...
Judging qualitative example 10/30 ...
Judging qualitative example 11/30 ...
Judging qualitative example 12/30 ...
Judging qualitative example 13/30 ...
Judging qualitative example 14/30 ...
Judging qualitative example 15/30 ...
Judging qualitative example 16/30 ...
Judging qualitative example 17/30 ...
Judging qualitative example 18/30 ...
Judging qualitative example 19/30 ...
Judging qualitative example 20/30 ...
Judging qualitative example 21/30 ...
Judging qualitative example 22/30 ...
Judging qualitative example 23/30 ...
Judging qualitative example 24/30 ...
Judging qualitative example 25/30 ...
Judging qualitative example 26/30 ...
Judging qualitative e

,qualitative_example_id,test_row_id,article_tokens,source_article,reference_summary,model,model_summary,fluency_1to5,factual_consistency_1to5,coverage_1to5,conciseness_1to5,primary_error_type,notes,deepseek_rationale,judge_model
0,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,Lead-3,Women have long been warned that the older the...,5,5,4,4,none,NaN,The summary is fluent and factually consistent...,deepseek-chat
1,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,"Fine-tuned BART-base, 1024-token truncation","Two-thirds of new UK fathers are now over 30, ...",4,5,3,3,too_verbose,NaN,The summary includes redundant information and...,deepseek-chat
2,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,Hierarchical BART,"Two-thirds of new UK fathers are now over 30, ...",4,5,2,2,too_verbose,NaN,The summary is overly verbose and includes min...,deepseek-chat
3,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,Fine-tuned LED-base-16384 CNN/DM,men over 35 have a 50% lower chance of conceiv...,5,5,3,5,omission,NaN,The summary is fluent and concise but omits th...,deepseek-chat
4,1,916,1134,Women have long been warned that the older the...,Two-thirds of new UK fathers are now over 30 ....,BART-large-CNN reference,"Two-thirds of new UK fathers are now over 30, ...",5,5,4,4,none,NaN,The summary is fluent and factually consistent...,deepseek-chat
5,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,Lead-3,The mother who crashed a car into a lake near ...,5,5,3,4,omission,NaN,"The summary is fluent and factually accurate, ...",deepseek-chat
6,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,"Fine-tuned BART-base, 1024-token truncation","Akon Guode, 35, was behind the wheel of a grey...",4,5,4,3,too_verbose,NaN,The summary is factually consistent and covers...,deepseek-chat
7,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,Hierarchical BART,"Akon Guode, 35, was behind the wheel of a grey...",4,5,4,3,too_verbose,NaN,"Factually accurate and covers key events, but ...",deepseek-chat
8,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,Fine-tuned LED-base-16384 CNN/DM,the four-wheel-drive plunged into the lake at ...,3,4,3,4,entity_error,NaN,"Contains minor factual errors (e.g., 'inner-we...",deepseek-chat
9,2,249,1468,The mother who crashed a car into a lake near ...,Car crashed into Melbourne lake just before 4p...,BART-large-CNN reference,The mother who crashed a car into a lake near ...,5,5,4,3,too_verbose,NaN,"Highly fluent and factually accurate, but incl...",deepseek-chat


## DeepSeek Judge Summary Tables

The following cell aggregates DeepSeek's rubric scores by method and counts primary error types. These are the tables to use when discussing LLM-judge qualitative results.


In [6]:
deepseek_scored = pd.read_csv(DEEPSEEK_SCORED_CSV)
for col in score_cols:
    deepseek_scored[col] = pd.to_numeric(deepseek_scored[col], errors='coerce')

deepseek_method_summary = deepseek_scored.groupby('model')[score_cols].mean().round(3)
deepseek_method_summary['overall_mean'] = deepseek_method_summary[score_cols].mean(axis=1).round(3)
deepseek_method_summary['n_outputs'] = deepseek_scored.groupby('model').size()
deepseek_method_summary = deepseek_method_summary.sort_values('overall_mean', ascending=False)

rank_df = deepseek_method_summary[score_cols + ['overall_mean']].rank(ascending=False, method='min').astype('Int64')
rank_df.columns = [f'{col}_rank' for col in rank_df.columns]
deepseek_method_comparison = deepseek_method_summary.join(rank_df)

for baseline in ['Lead-3', 'Fine-tuned BART-base, 1024-token truncation']:
    if baseline in deepseek_method_summary.index:
        for col in score_cols + ['overall_mean']:
            deepseek_method_comparison[f'{col}_minus_{baseline}'] = (deepseek_method_summary[col] - deepseek_method_summary.loc[baseline, col]).round(3)

deepseek_method_comparison.to_csv(DEEPSEEK_METHOD_SUMMARY_PATH)
print('Saved DeepSeek method comparison:', DEEPSEEK_METHOD_SUMMARY_PATH)
display(deepseek_method_comparison)

deepseek_error_summary = (
    deepseek_scored.assign(primary_error_type=deepseek_scored['primary_error_type'].fillna('unspecified'))
    .groupby(['model', 'primary_error_type'])
    .size()
    .reset_index(name='count')
    .sort_values(['model', 'count'], ascending=[True, False])
)
deepseek_error_summary.to_csv(DEEPSEEK_ERROR_SUMMARY_PATH, index=False)
print('Saved DeepSeek error-type summary:', DEEPSEEK_ERROR_SUMMARY_PATH)
display(deepseek_error_summary)


Saved DeepSeek method comparison: /content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2/results/deepseek_method_comparison_summary.csv


,fluency_1to5,factual_consistency_1to5,coverage_1to5,conciseness_1to5,overall_mean,n_outputs,fluency_1to5_rank,factual_consistency_1to5_rank,coverage_1to5_rank,conciseness_1to5_rank,...,fluency_1to5_minus_Lead-3,factual_consistency_1to5_minus_Lead-3,coverage_1to5_minus_Lead-3,conciseness_1to5_minus_Lead-3,overall_mean_minus_Lead-3,"fluency_1to5_minus_Fine-tuned BART-base, 1024-token truncation","factual_consistency_1to5_minus_Fine-tuned BART-base, 1024-token truncation","coverage_1to5_minus_Fine-tuned BART-base, 1024-token truncation","conciseness_1to5_minus_Fine-tuned BART-base, 1024-token truncation","overall_mean_minus_Fine-tuned BART-base, 1024-token truncation"
model,,,,,,,,,,,,,,,,,,,,,
BART-large-CNN reference,4.867,4.967,3.567,3.933,4.334,30,1,2,1,1,...,0.067,-0.033,1.067,0.533,0.409,0.500,0.200,0.767,0.566,0.509
Lead-3,4.800,5.000,2.500,3.400,3.925,30,2,1,3,3,...,0.000,0.000,0.000,0.000,0.000,0.433,0.233,-0.300,0.033,0.100
"Fine-tuned BART-base, 1024-token truncation",4.367,4.767,2.800,3.367,3.825,30,3,3,2,4,...,-0.433,-0.233,0.300,-0.033,-0.100,0.000,0.000,0.000,0.000,0.000
Hierarchical BART,4.267,4.633,2.500,3.600,3.750,30,4,4,3,2,...,-0.533,-0.367,0.000,0.200,-0.175,-0.100,-0.134,-0.300,0.233,-0.075
Fine-tuned LED-base-16384 CNN/DM,3.200,3.800,2.267,3.033,3.075,30,5,5,5,5,...,-1.600,-1.200,-0.233,-0.367,-0.850,-1.167,-0.967,-0.533,-0.334,-0.750


Saved DeepSeek error-type summary: /content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2/results/deepseek_error_type_summary.csv


,model,primary_error_type,count
2,BART-large-CNN reference,omission,14
1,BART-large-CNN reference,none,12
3,BART-large-CNN reference,too_verbose,3
0,BART-large-CNN reference,entity_error,1
7,"Fine-tuned BART-base, 1024-token truncation",omission,18
8,"Fine-tuned BART-base, 1024-token truncation",too_verbose,8
5,"Fine-tuned BART-base, 1024-token truncation",hallucination,2
4,"Fine-tuned BART-base, 1024-token truncation",entity_error,1
6,"Fine-tuned BART-base, 1024-token truncation",none,1
10,Fine-tuned LED-base-16384 CNN/DM,hallucination,9
